# Errors, logging and testing

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python-07-errors-logging-and-testing.ipynb)

A pipeline that fails silently is worse than one that fails loudly. This chapter covers the three habits that make
Python code trustworthy in production: raise specific errors and handle only what you can handle, log what happened in
a form people can search, and test transformations with small, fast checks.

## Exceptions: catch narrowly

`try` runs code that may fail. `except` handles a specific exception type. Catch the narrowest type you expect. A bare
`except:` or `except Exception: pass` hides bugs, including the ones you did not expect.

`finally` runs whether or not an error happened. `else` runs only when no error happened.

### Where you'll use it

- **Transient network errors:** catch `TimeoutError` and retry; let every other error stop the job so a real bug is not hidden.
- **Parsing with fallback:** catch the one error that means "bad text" and count it as a quarantined row.

In [ ]:
def parse_amount(text):
    try:
        value = float(text)
    except ValueError:
        return None            # expected: the source sent bad text
    else:
        return value
    finally:
        pass                   # runs every time; here it only shows the syntax

assert parse_amount("12.5") == 12.5
assert parse_amount("n/a") is None

## Raising and wrapping errors

Raise a specific error with a clear message. Define your own exception classes for the failures your pipeline cares
about, so callers can catch them by name. When you wrap a lower-level error, use `raise ... from ...` to keep the
original cause in the traceback.

### Where you'll use it

- **Schema contracts:** a `SchemaError` raised by the loader tells the on-call engineer exactly which columns drifted.
- **Adding context:** wrapping a driver error with the file name and row number, while keeping the original traceback.

In [ ]:
class SchemaError(Exception):
    """A record does not match the expected columns."""

def require_columns(record, required):
    missing = [column for column in required if column not in record]
    if missing:
        raise SchemaError(f"missing columns: {', '.join(missing)}")
    return record

try:
    require_columns({"id": 1}, ["id", "amount"])
except SchemaError as error:
    message = str(error)

assert message == "missing columns: amount"

def load_amount(record):
    try:
        return float(record["amount"])
    except (KeyError, ValueError) as error:
        raise SchemaError("bad amount") from error

try:
    load_amount({"amount": "x"})
except SchemaError as error:
    assert isinstance(error.__cause__, ValueError)   # the original cause is kept

## Logging instead of `print`

The `logging` module records messages with a level, a time and a logger name. Levels let you filter: `DEBUG` for detail,
`INFO` for progress, `WARNING` for something odd, `ERROR` for a failed step. Use `%s` placeholders, not f-strings, so
a message is formatted only if it is actually emitted.

Use `logger.exception` inside an `except` block to log the traceback too.

### Where you'll use it

- **Operating pipelines:** logs are how an engineer learns why a 3 a.m. run loaded 0 rows. Structured INFO lines with counts answer that question.
- **Alerting:** a monitoring tool counts `ERROR` lines and pages on-call when they pass a threshold.

In [ ]:
import io
import logging

stream = io.StringIO()
logger = logging.getLogger("orders.load")
logger.handlers.clear()
handler = logging.StreamHandler(stream)
handler.setFormatter(logging.Formatter("%(levelname)s %(name)s: %(message)s"))
logger.addHandler(handler)
logger.setLevel(logging.INFO)

logger.debug("hidden: below the level")
logger.info("loaded %d rows from %s", 120, "orders.csv")
logger.warning("skipped %d bad rows", 3)

try:
    load_amount({"amount": "x"})
except SchemaError:
    logger.exception("could not load a record")

lines = stream.getvalue().splitlines()
assert lines[0] == "INFO orders.load: loaded 120 rows from orders.csv"
assert lines[1] == "WARNING orders.load: skipped 3 bad rows"
assert lines[2].startswith("ERROR orders.load: could not load a record")
assert not any("hidden" in line for line in lines)

## Testing transformations

A transformation is a function from input to output, which makes it easy to test. Test the behaviour you care about:
the normal case, the empty input, the edge values and the errors. Keep each check small and independent.

In a notebook, a test is a function whose body is `assert` statements. A test runner such as `pytest` collects the
functions named `test_*` in a file and reports every failure. The same functions work in both places.

### Where you'll use it

- **CI for data code:** a test suite runs on every pull request, so a change to a parsing rule breaks the build instead of production.
- **Regression tests:** when a bad record reaches production, add it to the test file so it cannot return.

In [ ]:
def normalise_country(code):
    if not code or not code.strip():
        raise ValueError("country code is empty")
    return code.strip().upper()

def test_normalise_country_strips_and_uppercases():
    assert normalise_country(" de ") == "DE"

def test_normalise_country_rejects_blank():
    try:
        normalise_country("   ")
    except ValueError:
        return
    raise AssertionError("expected ValueError")

def test_normalise_country_is_idempotent():
    once = normalise_country("nl")
    assert normalise_country(once) == once

for test in [test_normalise_country_strips_and_uppercases,
             test_normalise_country_rejects_blank,
             test_normalise_country_is_idempotent]:
    test()
print("all tests passed")

## Measuring time

`time.perf_counter()` is the clock to use for durations: it is monotonic and high resolution. `time.time()` can jump
when the system clock changes. A context manager makes timing reusable.

### Where you'll use it

- **Finding slow steps:** timing each stage of a job shows which one to optimise first.
- **Benchmarks:** comparing two parsing approaches on a sample file before choosing one.

In [ ]:
import time
from contextlib import contextmanager

@contextmanager
def timed(label, log):
    start = time.perf_counter()
    try:
        yield
    finally:
        log.append((label, time.perf_counter() - start))

timings = []
with timed("sum", timings):
    sum(range(1_000_000))
assert timings[0][0] == "sum" and timings[0][1] >= 0

## Practice questions

1. Write `safe_int(text, default)` that returns the integer or `default`, catching only the errors `int` raises for bad text.
2. Add a `retry_count` check: a function `flaky(state)` raises `ConnectionError` on its first two calls and returns `"ok"`
   after that. Write a loop that calls it up to 3 times and returns the result, or re-raises the last error.
3. Write a test for `safe_int` that checks it returns the default for `"12.5"`.

## Answers

### Question 1

In [ ]:
def safe_int(text, default):
    try:
        return int(text)
    except (TypeError, ValueError):
        return default

assert safe_int("42", 0) == 42
assert safe_int("x", -1) == -1
assert safe_int(None, 0) == 0

### Question 2

In [ ]:
def call_with_retries(fn, attempts, state):
    last_error = None
    for _ in range(attempts):
        try:
            return fn(state)
        except ConnectionError as error:
            last_error = error
    raise last_error

def flaky(state):
    state["calls"] += 1
    if state["calls"] <= 2:
        raise ConnectionError("temporary")
    return "ok"

state = {"calls": 0}
assert call_with_retries(flaky, 3, state) == "ok"
assert state["calls"] == 3

### Question 3

In [ ]:
def test_safe_int_returns_default_for_decimal_text():
    assert safe_int("12.5", 0) == 0

test_safe_int_returns_default_for_decimal_text()

## Interview notes

- **Catch narrowly, raise specifically.** Every `except Exception: pass` is a bug waiting for a day it hides data loss.
- **`raise ... from error` keeps the cause.** The traceback shows both errors, which speeds up debugging.
- **Log with `%s` placeholders and levels.** Logs are data: searchable, filterable and the first thing read after an incident.
- **Test behaviour, not implementation.** A test should still pass after a refactor that keeps the behaviour the same.